# 数据调用模板 · Stanford Dogs 64x64

**模型写好后，怎么把数据喂进去。** 本 notebook 全部单元格可直接运行（含冒烟测试）。

## 数据长什么样

```python
PT = r"...\Kaggle\data\stanford_dogs_64.pt"
# {'X': uint8 (20580, 3, 64, 64)  值域 0~255
#  'Y': int64 (20580,)           值域 0~119
#  'classes': list[str] 长度 120}
```

> ⚠️ **关键**：`X` 是 `uint8` 0~255，**不是** 0~1。训练时必须在线归一化：
> ```python
> xb.float().div(127.5).sub(1)     # 0~255 -> [-1, 1]
> ```

In [1]:
import os, sys, time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split

PROJECT = r"C:\Users\moneyforever\Desktop\Deep-Learning\Kaggle"
PT      = os.path.join(PROJECT, "data", "stanford_dogs_64.pt")
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device :', device)
print('torch  :', torch.__version__)

# ---------- 1. 读入 ----------
d = torch.load(PT, weights_only=False)
X, Y, classes = d['X'], d['Y'], d['classes']
print('X      :', X.dtype, tuple(X.shape), '值域', int(X.min()), '~', int(X.max()))
print('Y      :', Y.dtype, tuple(Y.shape), '值域', int(Y.min()), '~', int(Y.max()))
print('classes:', len(classes), '个  ->', classes[:3], '...')

device : cuda
torch  : 2.11.0+cu128
X      : torch.uint8 (20580, 3, 64, 64) 值域 0 ~ 255
Y      : torch.int64 (20580,) 值域 0 ~ 119
classes: 120 个  -> ['n02085620-Chihuahua', 'n02085782-Japanese_spaniel', 'n02085936-Maltese_dog'] ...


In [2]:
# ---------- 2. 划分 + DataLoader ----------
full = TensorDataset(X, Y)
n = len(full)
n_train = int(0.8 * n)

# 固定种子 -> 每次划分一致（做实验对比时很重要）
g = torch.Generator().manual_seed(42)
train_ds, val_ds = random_split(full, [n_train, n - n_train], generator=g)

BATCH = 128
train_iter = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0)
val_iter   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)

print('总样本 %d  |  训练 %d  |  验证 %d' % (n, n_train, n - n_train))
print('类别数 %d  |  batch %d' % (len(classes), BATCH))
print('num_workers=0 —— 数据已在内存，多进程只会更慢')

总样本 20580  |  训练 16464  |  验证 4116
类别数 120  |  batch 128
num_workers=0 —— 数据已在内存，多进程只会更慢


In [3]:
# ---------- 3. 取一个 batch 验证 ----------
xb, yb = next(iter(train_iter))
print('原始     :', xb.dtype, tuple(xb.shape), '范围', int(xb.min()), '~', int(xb.max()))

x_norm = xb.to(device).float().div(127.5).sub(1)     # <<< 归一化就这一行
print('归一化后 :', x_norm.dtype, tuple(x_norm.shape), '范围 %.2f ~ %.2f' % (x_norm.min(), x_norm.max()))
print('标签     :', yb[:6].tolist(), '->', [classes[i] for i in yb[:6].tolist()])

原始     : torch.uint8 (128, 3, 64, 64) 范围 0 ~ 255


归一化后 : torch.float32 (128, 3, 64, 64) 范围 -1.00 ~ 1.00
标签     : [13, 72, 88, 118, 117, 60] -> ['n02088632-bluetick', 'n02104365-schipperke', 'n02107683-Bernese_mountain_dog', 'n02115913-dhole', 'n02115641-dingo', 'n02100583-vizsla']


## 用法 A · 分类任务（监督训练）

模型需要输出 `num_classes` 维 logits。

In [4]:
# 你的模型（这里用一个极小 CNN 占位）
class ToyNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1), nn.ReLU(), nn.MaxPool2d(2),   # 64 -> 16
            nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(), nn.MaxPool2d(2),  # 16 -> 4
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, num_classes),
        )
    def forward(self, x):
        return self.body(x)

net = ToyNet(len(classes)).to(device)
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(net.parameters(), lr=1e-3)

# ---------- 训练循环模板 ----------
def train_one_epoch(net, loader):
    net.train(); total_loss = 0.0; correct = 0; seen = 0
    for xb, yb in loader:
        xb = xb.to(device).float().div(127.5).sub(1)   # <<< 别忘归一化
        yb = yb.to(device)
        out  = net(xb)
        loss = loss_fn(out, yb)
        opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item() * yb.numel()
        correct += (out.argmax(1) == yb).sum().item()
        seen += yb.numel()
    return total_loss / seen, correct / seen

@torch.no_grad()
def evaluate(net, loader):
    net.eval(); correct = 0; seen = 0
    for xb, yb in loader:
        xb = xb.to(device).float().div(127.5).sub(1)
        yb = yb.to(device)
        correct += (net(xb).argmax(1) == yb).sum().item()
        seen += yb.numel()
    return correct / seen

# ---- 冒烟测试：只在 1 个 batch 上跑一次，确认数据管道通 ----
xb, yb = next(iter(train_iter))
xb = xb.to(device).float().div(127.5).sub(1); yb = yb.to(device)
out = net(xb)
print('输出 logits :', tuple(out.shape), ' (应为 (batch, %d))' % len(classes))
print('单批 loss   :', round(nn.functional.cross_entropy(out, yb).item(), 4))
print('冒烟测试通过 ✅  把 train_one_epoch/evaluate 套进你的 epoch 循环即可')

输出 logits : (128, 120)  (应为 (batch, 120))
单批 loss   : 4.7937
冒烟测试通过 ✅  把 train_one_epoch/evaluate 套进你的 epoch 循环即可


## 用法 B · GAN 训练（判别器只要真实图，不要标签）

`X` 直接丢掉 `Y` 即可。**必须 `drop_last=True`**，否则最后一个不满的 batch 会让 BatchNorm 报错。

In [5]:
# 只要图片
real_ds   = TensorDataset(X)
real_iter = DataLoader(real_ds, batch_size=64, shuffle=True,
                       num_workers=0, drop_last=True)

z_dim = 100
# 生成器：1x1 -> 4 -> 8 -> 16 -> 32 -> 64  (共 5 层上采样，最终 64x64)
G = nn.Sequential(
    nn.ConvTranspose2d(z_dim, 256, 4, 1, 0), nn.BatchNorm2d(256), nn.ReLU(),   # 4x4
    nn.ConvTranspose2d(256, 128, 4, 2, 1),   nn.BatchNorm2d(128), nn.ReLU(),   # 8x8
    nn.ConvTranspose2d(128, 64, 4, 2, 1),    nn.BatchNorm2d(64),  nn.ReLU(),   # 16x16
    nn.ConvTranspose2d(64, 32, 4, 2, 1),     nn.BatchNorm2d(32),  nn.ReLU(),   # 32x32
    nn.ConvTranspose2d(32, 3, 4, 2, 1),      nn.Tanh(),                        # 64x64, [-1,1]
).to(device)

# 判别器：用 AdaptiveAvgPool 收尾，对输入尺寸不敏感，避免写死维度
D = nn.Sequential(
    nn.Conv2d(3, 64, 4, 2, 1),   nn.LeakyReLU(0.2),            # 64 -> 32
    nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),  # 32 -> 16
    nn.Conv2d(128, 256, 4, 2, 1),nn.BatchNorm2d(256), nn.LeakyReLU(0.2),  # 16 -> 8
    nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(256, 1),
).to(device)

# ---- 冒烟测试：跑一个真实 batch ----
(x_real,) = next(iter(real_iter))
x_real = x_real.to(device).float().div(127.5).sub(1)      # <<< 真实图 -> [-1,1]
z      = torch.randn(x_real.size(0), z_dim, 1, 1, device=device)
x_fake = G(z)
print('真实图 :', tuple(x_real.shape), '范围 %.2f ~ %.2f' % (x_real.min(), x_real.max()))
xf = x_fake.detach()
print('生成图 :', tuple(xf.shape), '范围 %.2f ~ %.2f' % (xf.min(), xf.max()))
print('判别器 :', tuple(D(x_fake).shape))
print('冒烟测试通过 ✅')

真实图 : (64, 3, 64, 64) 范围 -1.00 ~ 1.00
生成图 : (64, 3, 64, 64) 范围 -1.00 ~ 1.00
判别器 : (64, 1)
冒烟测试通过 ✅


**为什么用 `[-1,1]`**：生成器末层是 `tanh`，输出就是 `[-1,1]`。真实图必须落在同一区间，否则判别器会靠"数值范围"作弊。

若你的生成器末层用 `sigmoid`（输出 `0~1`），把真实图换成 `xb.float().div(255)` 即可。

## 用法 C · 不预载内存的流式方案（等价）

如果以后换更大的数据集，`.pt` 占内存太多时用这个。`data\all-dogs` 联接已建好。

In [6]:
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),        # 直接得到 float 0~1
])

ds = datasets.ImageFolder(
    root=os.path.join(PROJECT, "data", "all-dogs"),   # junction -> images/Images
    transform=transform,
)
print('类别数:', len(ds.classes), '| 图片数:', len(ds))

stream_iter = DataLoader(ds, batch_size=128, shuffle=True, num_workers=4)  # 这时才需要 workers

xb, yb = next(iter(stream_iter))
x_norm = xb.to(device).float().div(0.5).sub(1)        # ToTensor 已是 0~1，故除 0.5 减 1
print('流式 batch :', tuple(xb.shape), '| 归一化范围 %.2f ~ %.2f' % (x_norm.min(), x_norm.max()))

# ---- 两种方案等价性验证 ----
pt_batch = next(iter(train_iter))[0].to(device).float().div(127.5).sub(1)
print()
print('两种方案归一化公式对照：')
print('  .pt  (uint8 0~255) : .float().div(127.5).sub(1)')
print('  流式 (float 0~1)   : .float().div(0.5).sub(1)')
print('  结果区间一致 ->', '%.2f ~ %.2f' % (pt_batch.min(), pt_batch.max()),
      'vs', '%.2f ~ %.2f' % (x_norm.min(), x_norm.max()))

类别数: 120 | 图片数: 20580


流式 batch : (128, 3, 64, 64) | 归一化范围 -1.00 ~ 1.00

两种方案归一化公式对照：
  .pt  (uint8 0~255) : .float().div(127.5).sub(1)
  流式 (float 0~1)   : .float().div(0.5).sub(1)
  结果区间一致 -> -1.00 ~ 1.00 vs -1.00 ~ 1.00


## 用法 D · 接项目里的 `gan_eval` 评估

> 下面这段**不执行**（首次会下载 Inception 权重），直接复制到你的评估 notebook 用。

```python
import os, sys
PROJECT = r"C:\Users\moneyforever\Desktop\Deep-Learning\Kaggle"
sys.path.insert(0, PROJECT)
from gan_eval import GANEvaluator, FIDTracker, list_images

REAL_DIR = os.path.join(PROJECT, "data", "all-dogs")        # 已建联接（真实参考集）
FAKE_DIR = os.path.join(PROJECT, "outputs", "generated")    # 你保存的生成图文件夹
OUT_DIR  = os.path.join(PROJECT, "outputs", "eval")

# 一次性算好真实图特征并缓存（真实集不变，不必重复算）
ev = GANEvaluator(real_dir=REAL_DIR, image_size=64, device=device)

# 评估：第一个参数是 fake（文件夹路径，或 uint8 张量 (N,3,64,64)）
# 合法指标键只有：fid / mifid / kid / is / pr
results = ev.evaluate(FAKE_DIR, metrics=("fid", "mifid", "kid", "is", "pr"))
print(ev.report(results))

# 训练时跟踪 FID 曲线
tracker = FIDTracker(ev, out_dir=OUT_DIR, every=5, num_samples=2000)

def sample_fn(n, seed):          # 你的生成器采样：返回 uint8 (n,3,64,64)
    ...

# 每个 epoch 末（epoch % tracker.every == 0 时）调用：
# tracker.step(epoch, sample_fn)     # 自动存图 + 算分 + 写 fid_history.json
# tracker.plot()                     # 画 FID / MiFID 曲线
```

命令行等价写法（脚本已移到 `scripts/`）：

```bash
python scripts/evaluate.py --real data/all-dogs --fake outputs/generated --image-size 64 --nn-grid
```


## 速查表

| 场景 | 取数据 | 归一化 |
|---|---|---|
| 分类（监督） | `TensorDataset(X, Y)` + `random_split` | `.float().div(127.5).sub(1)` |
| GAN（判别器） | `TensorDataset(X)`，`drop_last=True` | `.float().div(127.5).sub(1)` |
| 流式（省内存） | `ImageFolder("data/all-dogs")` | `.float().div(0.5).sub(1)` |

| 参数 | 值 |
|---|---|
| 图片尺寸 | 64×64 |
| 类别数 | 120 |
| 总样本 | 20,580（训练 16,464 / 验证 4,116） |
| 建议 batch | 分类 128，GAN 64 |
| `num_workers` | `.pt` 用 0，流式用 4 |